In [20]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

In [5]:
# 1. Cargar el dataset con tus 3.444 compuestos que superaron el umbral de 6.0
df_candidatos = pd.read_csv('../results/TOP_inhibidores_FabI_CMNPD.csv')

In [ ]:
# 2. Definir la función para evaluar la Regla de Lipinski
def lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False 
        
    # Calcular los 4 parámetros exactos usando el módulo Descriptors de RDKit
    peso_molecular = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    donadores_h = Descriptors.NumHDonors(mol)
    aceptores_h = Descriptors.NumHAcceptors(mol)

    # 3. Aplicar filtro estricto (0 violaciones permitidas)
    if (peso_molecular < 500) and (logp < 5) and (donadores_h <= 5) and (aceptores_h <= 10):
        return True
    else:
        return False

print("Calculando parámetros de Lipinski para todos los compuestos...")

In [13]:
# 4. Aplicar la función a la columna SMILES
df_candidatos['Reglas_Lipinski'] = df_candidatos['Smiles'].apply(lipinski)

In [ ]:
# 5. Filtrar el DataFrame quedándonos solo con los que cumplen la regla
df_lipinski = df_candidatos[df_candidatos['Reglas_Lipinski'] == True].copy()
df_lipinski = df_lipinski.drop(columns=['Reglas_Lipinski']).reset_index(drop=True)

,Smiles,pChEMBL_Fisico,pChEMBL_MACCS,Media_Predicciones


In [9]:
# 6. Guardar los compuestos viables como medicamentos orales
df_lipinski.to_csv('../results/CMNPD_Lipinski_Aprobados.csv', index=False)

print(f"\n¡Filtrado completado! De {len(df_candidatos)} candidatos iniciales, {len(df_lipinski)} cumplen estrictamente la Regla de Lipinski.")
df_lipinski.head()


¡Filtrado completado! De 3444 candidatos iniciales, 0 cumplen estrictamente la Regla de Lipinski.


,Smiles,pChEMBL_Fisico,pChEMBL_MACCS,Media_Predicciones


In [14]:
# 2. Definir una función de Lipinski más realista (contando violaciones)
def calcular_violaciones_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 99 # Valor alto para descartar si el SMILES es inválido
        
    violaciones = 0
    
    if Descriptors.MolWt(mol) > 500:
        violaciones += 1
    if Descriptors.MolLogP(mol) > 5:
        violaciones += 1
    if Descriptors.NumHDonors(mol) > 5:
        violaciones += 1
    if Descriptors.NumHAcceptors(mol) > 10:
        violaciones += 1
        
    return violaciones

print("Calculando el número de violaciones de Lipinski por compuesto...")

Calculando el número de violaciones de Lipinski por compuesto...


In [15]:
# 3. Aplicar la función
df_candidatos['Violaciones_Lipinski'] = df_candidatos['Smiles'].apply(calcular_violaciones_lipinski)

In [16]:
# 4. APLICAR FILTRO FLEXIBLE: Nos quedamos con los que tienen 1 violación o menos (puedes poner 2 si siguen siendo pocos)
max_violaciones_permitidas = 1
df_lipinski = df_candidatos[df_candidatos['Violaciones_Lipinski'] <= max_violaciones_permitidas].copy()

In [17]:
# Opcional: limpiar la columna y reiniciar el índice
df_lipinski = df_lipinski.drop(columns=['Violaciones_Lipinski']).reset_index(drop=True)


In [19]:
# 5. Guardar los compuestos
df_lipinski.to_csv('../results/CMNPD_Lipinski_Flexible.csv', index=False)

print(f"\n¡Filtrado completado! De {len(df_candidatos)} candidatos iniciales, {len(df_lipinski)} cumplen la regla permitiendo hasta {max_violaciones_permitidas} violación(es).")


¡Filtrado completado! De 3444 candidatos iniciales, 1239 cumplen la regla permitiendo hasta 1 violación(es).


Selección de Scaffolds

In [21]:
print("Extrayendo esqueletos de Murcko para agrupar por familias...")

# 1. Función para extraer el andamiaje (scaffold) de cada molécula
def obtener_scaffold(smiles):
    try:
        # Genera el esqueleto base en formato SMILES
        return MurckoScaffold.MurckoScaffoldSmilesFromSmiles(smiles)
    except:
        return None

Extrayendo esqueletos de Murcko para agrupar por familias...


In [22]:
# 2. Aplicar la función a los compuestos que pasaron Lipinski
df_lipinski['Scaffold'] = df_lipinski['Smiles'].apply(obtener_scaffold)
# Limpiar posibles errores
df_lipinski = df_lipinski.dropna(subset=['Scaffold'])

In [23]:
# 3. ORDENAR Y FILTRAR: 
# Asegurarnos de que están ordenados por su predicción de mejor a peor
df_lipinski = df_lipinski.sort_values(by='Media_Predicciones', ascending=False)

In [24]:
# 4. Eliminar duplicados basándonos en el Scaffold, conservando el primero (el de mayor puntuación)
df_representantes = df_lipinski.drop_duplicates(subset=['Scaffold'], keep='first').reset_index(drop=True)
print(f"De los 1.239 compuestos, se han identificado {len(df_representantes)} familias (scaffolds) únicas.")
df_representantes[['Smiles', 'Scaffold', 'Media_Predicciones']].head(10)

De los 1.239 compuestos, se han identificado 519 familias (scaffolds) únicas.


,Smiles,Scaffold,Media_Predicciones
0,CC[C@H]1C=CCC[C@@]2(CC3CCc4c(C(=O)OCCCCCCCCCCC...,C1=CCOC2(CC1)CC1CCc3ccnc([n+]31)N2,6.620685
1,COC1=C(Br)C[C@]2(OC=C1Br)ON=C(C(=O)NCCCOc1c(Br...,O=C(NCCCOc1ccccc1)C1=NOC2(CC=CC=CO2)C1,6.568743
2,CCC(=O)N(C)CCc1cc(Br)c(OCCCNC(=O)Cc2ccc(O)cc2)...,O=C(Cc1ccccc1)NCCCOc1ccccc1,6.518519
3,COC1=C(Br)[C@@H](O)[C@]2(C=C1Br)CC(C(=O)NCCCOc...,O=C(NCCCOc1ccccc1)C1=NOC2(C=CC=CC2)C1,6.500993
4,CC(=O)N(C)CCc1cc(Br)c(OCCCNc2ncn(C)c3ncnc2-3)c...,c1ccc(OCCCNc2nc[nH]c3ncnc2-3)cc1,6.481907
5,C[S+]([O-])CC[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@...,O=C1NCC(=O)N2CCCC2C(=O)N2CCCC2C(=O)NCC(=O)N2CC...,6.481906
6,COC1=C(Br)[C@H](O)[C@@]2(C=C1Br)CC(C(=O)NCCc1c...,O=C(NCCc1ccccc1)C1=NOC2(C=CC=CC2)C1,6.480598
7,CC[C@H](C)[C@@H](C(=O)N[C@H](C(=O)N(C)[C@@H](C...,O=C(CCC1CCCN1)OCCc1ccccc1,6.476596
8,CC[C@H](C)[C@@H]([C@@H](CC(=O)N1CCC[C@H]1[C@H]...,O=C(CCC1CCCN1)OCCCc1ccccc1,6.475787
9,CN[C@@H](Cc1ccc(OC)c(Br)c1)C(=O)NCCc1cc(Br)c(O...,O=C(CCc1ccccc1)NCCc1ccccc1,6.460417


In [25]:
# 5. Guardar nuestra selección final élite
df_representantes.to_csv('../results/CMNPD_Representantes_Elite.csv', index=False)